# Lab 6: Word Sense Disambiguation

##### Autors: Josep de Cid & Gonzalo Recio

In [1]:
from nltk.wsd import lesk
import nltk
from nltk import pos_tag
from nltk.metrics import jaccard_distance
from scipy.stats import pearsonr
import termtables as tt
from nltk.stem import WordNetLemmatizer
import re
from nltk.tokenize import RegexpTokenizer

context = ['I', 'went', 'to', 'the', 'bank', 'to', 'deposit', 'money', '.']

synset = lesk(context, 'bank', 'n')

str(synset)

"Synset('savings_bank.n.02')"

First we read the dataset into a list of pairs containing the sentences *[(A1, B1), ..., (An, Bn)]*

In [2]:
sentences = []

with open('trial/STS.input.txt', mode='r') as f:
    for idx, line in enumerate(f.readlines()):
        _, a, b = line.strip().split('\t')
        sentences.append([a, b])
        
        print(f'{idx + 1}a) {a}')
        print(f'{idx + 1}b) {b}\n')

1a) The bird is bathing in the sink.
1b) Birdie is washing itself in the water basin.

2a) In May 2010, the troops attempted to invade Kabul.
2b) The US army invaded Kabul on May 7th last year, 2010.

3a) John said he is considered a witness but not a suspect.
3b) "He is not a suspect anymore." John said.

4a) They flew out of the nest in groups.
4b) They flew into the nest together.

5a) The woman is playing the violin.
5b) The young lady enjoys listening to the guitar.

6a) John went horse back riding at dawn with a whole group of friends.
6b) Sunrise at dawn is a magnificent view to take in if you wake up early enough for it.



We generate a dict to translate PoS tagging from Penn Treebank format to nltk format.

In [3]:
pos_translation = {
    'NN': 'n',
    'NNP': 'n',
    'NNS': 'n',
    'NNPS': 'n',
    'VB': 'v',
    'VBD': 'v',
    'VBG': 'v',
    'VBN': 'v',
    'VBP': 'v',
    'VBZ': 'v',
    'JJ': 'a',
    'JJR': 'a',
    'JJS': 'a',
    'RB': 'r',
    'RBR': 'r',
    'RBS': 'r'   
}

Function to get all the synsets of the words of a given sentence

In [4]:
# Function to get all the synsets of the words of a given sentence
def get_senses_from_sentence(sentence, use_definitions=False):
    synsets = ''
    context = nltk.word_tokenize(sentence)
    pos_list = pos_tag(context)
    #         pos = pos_tag([word])[0][1]
        
    for word, pos in pos_list:
        # if it is an open-class world
        if pos in pos_translation:
            synset = lesk(context, word, pos=pos_translation[pos])
            if synset: 
                # print(synset.name())
                synset_name =synset.name().split(".")[0] 
                if use_definitions:
                    synsets += f' {synset.definition()}'
                else:
                    synsets += f' {synset_name}' 
            
    return synsets

In [5]:
def jaccard_distance_sentences(pair, lang='english'):
    print('Intersection:', nltk.word_tokenize(pair[0], language=lang))
    return jaccard_distance(
        set(nltk.word_tokenize(pair[0], language=lang)),
        set(nltk.word_tokenize(pair[1], language=lang))
    )

Compute distances using synsets obtained from the two sentences

In [6]:
distances = []

for s1, s2 in sentences:
    def1 = get_senses_from_sentence(s1)
    def2 = get_senses_from_sentence(s2)
    print()
    print('Synsets sentence 1:',def1, '\nSynsets sentence 2:', def2)
    distance = jaccard_distance_sentences((def1, def2))
    distances.append(distance)
    print('Jaccard distance:', distance)
    print()
    


Synsets sentence 1:  bird be bathe sinkhole 
Synsets sentence 2:  shuttlecock be wash body_of_water washbasin
Intersection: ['bird', 'be', 'bathe', 'sinkhole']
Jaccard distance: 0.875


Synsets sentence 1:  whitethorn troop undertake invade kabul 
Synsets sentence 2:  uranium united_states_army invade kabul whitethorn last year
Intersection: ['whitethorn', 'troop', 'undertake', 'invade', 'kabul']
Jaccard distance: 0.6666666666666666


Synsets sentence 1:  whoremaster suppose embody view witness not defendant 
Synsets sentence 2:  embody not defendant anymore whoremaster suppose
Intersection: ['whoremaster', 'suppose', 'embody', 'view', 'witness', 'not', 'defendant']
Jaccard distance: 0.375


Synsets sentence 1:  fly group 
Synsets sentence 2:  fly together
Intersection: ['fly', 'group']
Jaccard distance: 0.6666666666666666


Synsets sentence 1:  woman be play violin 
Synsets sentence 2:  young lady love heed guitar
Intersection: ['woman', 'be', 'play', 'violin']
Jaccard distance: 1.0


## Results S2 (using only words intersection)

In [7]:
# Distances obtained from previous session
S2_distances =  [0.6923076923076923,
                 0.7368421052631579,
                 0.5333333333333333,
                 0.5454545454545454,
                 0.7692307692307693,
                 0.8620689655172413]

## Results S3 (using lemmas)

In [8]:
# Distances obtained from previous session
S3_distances =  [0.6923076923076923,
                 0.6666666666666666,
                 0.5333333333333333,
                 0.5454545454545454,
                 0.7692307692307693,
                 0.8620689655172413]

## Comparison of previous sessions with S6 (using word sense desambiguation - synset Jaccard distance)

In [9]:
refs = [0, 1, 2, 3, 4, 5]  # Golden standard

# Print distances from all sessions to compare results
print(tt.to_string(
        data=[['S2 distances']+[f'{x:.5}'for x in S2_distances],
              ['S3 distances']+[f'{x:.5}'for x in S3_distances],
              ['S6 distances']+[f'{x:.5}'for x in distances]],
        header=[' ', 'pair 1', 'pair 2','pair 3','pair 4','pair 5','pair 6']
            
))

print()
# Print correlations with golden standard
print(f'S2 correlation: {pearsonr(refs, S2_distances)[0]:.15}')
print(f'S3 correlation: {pearsonr(refs, S3_distances)[0]:.15}')
print(f'S6 correlation: {pearsonr(refs, distances)[0]:.15}')

┌──────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│              │ pair 1  │ pair 2  │ pair 3  │ pair 4  │ pair 5  │ pair 6  │
╞══════════════╪═════════╪═════════╪═════════╪═════════╪═════════╪═════════╡
│ S2 distances │ 0.69231 │ 0.73684 │ 0.53333 │ 0.54545 │ 0.76923 │ 0.86207 │
├──────────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ S3 distances │ 0.69231 │ 0.66667 │ 0.53333 │ 0.54545 │ 0.76923 │ 0.86207 │
├──────────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ S6 distances │ 0.875   │ 0.66667 │ 0.375   │ 0.66667 │ 1.0     │ 0.9375  │
└──────────────┴─────────┴─────────┴─────────┴─────────┴─────────┴─────────┘

S2 correlation: 0.396238977611923
S3 correlation: 0.490670810375692
S6 correlation: 0.370559360149974


We can see that the similarity performance does not improve much using WSD.

### Distance using synsets definitions
Before analysing the results, we will try to see if we can improve a bit the obtained results. Now we try to compute the Jaccard distance with the **definitions** of the synsets (removing stopwords and lemmatizing words) instead of just the synset names. Perhaps using the definitions we manage to detect more similarities that with synset names we couldn't detect.

Needed code to preprocess and lemmatize definitions:

In [10]:
from nltk.corpus import stopwords
sw = set(stopwords.words('english'))
wnl = WordNetLemmatizer()
tokenizer = RegexpTokenizer(r'\w+')

def preprocess_sentence(s):
    s = re.sub(r'[0-9]+', '', s).strip()  # remove digits
    s = s.lower()                         # to lowercase
    s = ' '.join([w.lower() for w in tokenizer.tokenize(s) if w.lower() not in sw])   
    # removes double spacing and strange punctuation chars
    return s

def tag_sentences(sentence, lang='english'):
    sentence = preprocess_sentence(sentence)
    return pos_tag(nltk.word_tokenize(sentence, language=lang))

def lemmatize_token(token):
    if token[1][0] in ['N', 'V', 'J', 'R']:
        return wnl.lemmatize(token[0].lower(), pos=pos_translation[token[1]])
    return token[0]

def lemmatize_sentence(sentence_pos):
    return list(map(lemmatize_token, sentence_pos))

Compute distances with definitions

In [11]:
distances_def = []

for s1, s2 in sentences:
    def1 = get_senses_from_sentence(s1, use_definitions=True)
    def2 = get_senses_from_sentence(s2, use_definitions=True)

    def1 = ' '.join(lemmatize_sentence(tag_sentences(def1)))
    def2 = ' '.join(lemmatize_sentence(tag_sentences(def2)))
    print()
    distance = jaccard_distance_sentences((def1, def2))
    distances_def.append(distance)
    print('Jaccard distance:', distance)
    print()


Intersection: ['flesh', 'bird', 'fowl', 'wild', 'domestic', 'use', 'food', 'remain', 'unmolested', 'undisturbed', 'uninterrupted', 'use', 'infinitive', 'form', 'cleanse', 'entire', 'body', 'depression', 'ground', 'communicate', 'subterranean', 'passage', 'especially', 'limestone', 'formed', 'solution', 'collapse', 'cavern', 'roof']
Jaccard distance: 0.8833333333333333


Intersection: ['thorny', 'eurasian', 'shrub', 'small', 'tree', 'dense', 'cluster', 'white', 'scarlet', 'flower', 'follow', 'deep', 'red', 'berry', 'establish', 'escape', 'eastern', 'north', 'america', 'cavalry', 'unit', 'correspond', 'infantry', 'company', 'enter', 'upon', 'activity', 'enterprise', 'march', 'aggressively', 'another', 'territory', 'military', 'force', 'purpose', 'conquest', 'occupation', 'capital', 'large', 'city', 'afghanistan', 'locate', 'eastern', 'afghanistan']
Jaccard distance: 0.691358024691358


Intersection: ['prostitute', 'customer', 'express', 'supposition', 'represent', 'character', 'stage', 

Results of the distances using synset definitions

In [12]:
print('S6 using definitions:')
print(tt.to_string([[f'{i:.5}' for i in distances_def]],
                   alignment='cccccc',
                   header=['pair 1', 'pair 2','pair 3','pair 4','pair 5','pair 6']
                  )
     )
print(f'S6 correlation: {pearsonr(refs, distances_def)[0]:.15}')

S6 using definitions:
┌─────────┬─────────┬─────────┬────────┬─────────┬─────────┐
│ pair 1  │ pair 2  │ pair 3  │ pair 4 │ pair 5  │ pair 6  │
╞═════════╪═════════╪═════════╪════════╪═════════╪═════════╡
│ 0.88333 │ 0.69136 │ 0.35714 │  1.0   │ 0.89091 │ 0.98925 │
└─────────┴─────────┴─────────┴────────┴─────────┴─────────┘
S6 correlation: 0.387175693063167


# Conclusions

We haven't seen an improvement using the Word Sense Desambiguation to measure the distances between sentence pairs. First we tried to compute Jaccard distance using the synsets obtained from Lesk algorithm but the correlation with GS obtained was of **0.37** (worse than S2 and S3). 

After that, we tried to compute the distance with synset definitions insted of just synsets; this idea was motivated because we observed that some synsets like **woman.n.01** and **lady.n.01** were the same refearing to the same thing synsets were completely different (maybe definitions shared some common information). Using the synset definitions (removing stopwords and lemmatizing) we obtained a correlation of **0.387**, not a big improvement.

The reason of not obtaining significant improvement for this task can be because Lesk algorithm is very sensitive to the definition of words and the words used in definition might not overlap with the context. 
